In [5]:
import sys
sys.path.insert(0, "../src")
from chunking_baselines import chunk_fixed, chunk_sliding, chunk_dynamic
import pandas as pd

# Load discharge notes — the file is in the project root as a .gz file
discharge = pd.read_csv("../data/mimic/processed/discharge.csv", low_memory=False)
print(f"Loaded {len(discharge):,} notes")

# Pick 3 sample notes
sample_notes = discharge["text"].dropna().sample(3, random_state=1).tolist()

# 3 sample queries
queries = [
    "what medications were prescribed at discharge?",
    "what was the patient's primary diagnosis?",
    "were there any complications during the hospital stay?"
]

for i, (note, query) in enumerate(zip(sample_notes, queries)):
    print(f"\n{'='*60}")
    print(f"QUERY {i+1}: {query}")
    print(f"{'='*60}")

    print("\n-- Baseline A (Fixed 512) --")
    a = chunk_fixed(note)
    print(f"  {len(a)} chunks | First: {a[0][:120]}...")

    print("\n-- Baseline B (Sliding 256) --")
    b = chunk_sliding(note)
    print(f"  {len(b)} chunks | First: {b[0][:120]}...")

    print("\n-- Baseline C (Dynamic) --")
    c = chunk_dynamic(note, query, top_n=3)
    print(f"  Top 3 returned | Best: {c[0][:120]}...")

Loaded 331,793 notes

QUERY 1: what medications were prescribed at discharge?

-- Baseline A (Fixed 512) --
  4 chunks | First: Name: ___ Unit No: ___ Admission Date: ___ Discharge Date: ___ Date of Birth: ___ Sex: M Service: MEDICINE Allergies: Pe...

-- Baseline B (Sliding 256) --
  12 chunks | First: Name: ___ Unit No: ___ Admission Date: ___ Discharge Date: ___ Date of Birth: ___ Sex: M Service: MEDICINE Allergies: Pe...

-- Baseline C (Dynamic) --
  Top 3 returned | Best: EVERY OTHER DAY (Every Other Day). 14. Warfarin Warfarin 3.5 mg PO every other day. 15. Hexavitamin Tablet Sig: One (1) ...

QUERY 2: what was the patient's primary diagnosis?

-- Baseline A (Fixed 512) --
  4 chunks | First: Name: ___ Unit No: ___ Admission Date: ___ Discharge Date: ___ Date of Birth: ___ Sex: F Service: MEDICINE Allergies: la...

-- Baseline B (Sliding 256) --
  13 chunks | First: Name: ___ Unit No: ___ Admission Date: ___ Discharge Date: ___ Date of Birth: ___ Sex: F Service: MEDICINE Allergie

## Observations — Chunking Strategy Comparison

### Key Finding
- **Baseline A (Fixed 512)** and **Baseline B (Sliding 256)** both return the 
  patient header as their first chunk for every query — they are not query-aware.
- **Baseline C (Dynamic)** selects chunks based on semantic similarity to the 
  query using ClinicalBERT embeddings. It consistently retrieves more relevant 
  content:
  - For medication query → retrieved actual medication list (Warfarin etc.)
  - For diagnosis query → retrieved discharge instructions section
  - For complications query → retrieved followup/changes section

### Implication for thesis
This demonstrates that retrieval-guided dynamic chunking outperforms fixed 
strategies for clinical question answering — supporting the dual-source RAG 
framework's design choice of query-aware retrieval.

## Farhana's Observations (Week 6)

### What I noticed when running the notebook:
- Confirmed Riktika's key finding: Baselines A and B are completely query-blind —
  they return the same chunks regardless of what you ask.
- Baseline C (Dynamic) felt noticeably smarter — the chunks it returned actually
  matched the topic of the query, not just the beginning of the note.

### Why this matters for our dual-source RAG:
- Our PMC literature retriever (which I'm building this week) also uses semantic
  similarity — so we are applying the same query-aware principle to the literature
  source as Baseline C applies to MIMIC notes.
- This consistency across both sources (patient notes + literature) is a core
  design strength of our framework.

### One thing to investigate later:
- Baseline C picks the top-N chunks by cosine similarity but does not consider
  the order of chunks in the original note. A future ablation (Week 15) could
  test whether preserving document order alongside similarity scores improves results.